# D1-14 Optional - Fast multiple LCAs

Use `bw` and the BAFU database imported in D1-04. Calculate **100 datasets × the available
EF v3.1 categories**. Build each characterization matrix once, then reuse it for each inventory.
The technosphere stays fixed throughout this example.

In [ ]:
import bw2data as bd
import bw2calc
import numpy as np
import pandas as pd

Each row represents one unit of a dataset's reference product; the row labels include its
location and unit. Each column is a separate indicator, with its impact unit shown in the
column label. Categories include sub-indicators: do not sum the columns.

In [ ]:
bd.projects.set_current('aalborg-rlcia-2026')
products = sorted(bd.Database('bafu'), key=lambda a: a['code'])[:100]
EFV3 = [method for method in bd.methods if method[0] == 'EF v3.1']

In [ ]:
def get_lcia_scores(products, categories, results):
    lca = bw2calc.LCA({products[0].id: 1}, categories[0])
    lca.lci(factorize=True)  # Reuse the technosphere factorization too.
    lca.lcia()
    method_matrices = [lca.characterization_matrix.copy()]

    for other_method in categories[1:]:
        # Build each characterization matrix only once.
        lca.switch_method(other_method)
        method_matrices.append(lca.characterization_matrix.copy())

    for i, product in enumerate(products):
        lca.redo_lci({product.id: 1})  # Repeated demands use integer IDs.
        for j, characterization_matrix in enumerate(method_matrices):
            results[i, j] = (characterization_matrix * lca.inventory).sum()
    return results

In [ ]:
%%time
results = np.zeros((len(products), len(EFV3)))

results = get_lcia_scores(products, EFV3, results)

pd.DataFrame(
    results,
    index=[f"{p['name']} | {p['location']} | 1 {p['unit']}" for p in products],
    columns=[f"{m[1]} [{bd.Method(m).metadata['unit']}]" for m in EFV3],
)